In [ ]:
using Pkg
using Metal

cd(@__DIR__)
Pkg.activate("../")
ParamFile = "../config/testparam.csv"  # maybe GeoPoints and planet1D should be fusioned

# batchGPU should be at this level (I have not made it as a module yet, since the choice of Metal/CUDA should be done in a manual way)
include("../src/batchFiles/batchGPU.jl")


include("../src/commonBatchs.jl")
include("../src/planet1D.jl")
include("../src/GeoPoints.jl")

include("../src/flexOPT.jl")

using .commonBatchs, .planet1D, .GeoPoints, .flexOPT

# configurations (OPTnumerically)

In [ ]:
# global parameter ranges
orderBspace_list = -1:2
pointsInSpace_list = 3:5
supplementaryOrder_list = 2:4

orderBtime = 1
pointsInTime = 1

function make_itpl_cases(pointsInSpace)
    cases = NamedTuple[]

    push!(cases, (
        ptsSpace = 1,
        ptsTime = 1,
        offsetSpace = 1.0,
        offsetTime = 1,
        YorderBspace = 1,
        YorderBtime = 1,
    ))

    for ptsSpace in 2:min(4, pointsInSpace)
        for YorderBspace in 1:2
            push!(cases, (
                ptsSpace = ptsSpace,
                ptsTime = 1,
                offsetSpace = 0.5,
                offsetTime = 1,
                YorderBspace = YorderBspace,
                YorderBtime = 1,
            ))
        end
    end

    return cases
end

all_Configcases = [
    (
        orderBtime = orderBtime,
        orderBspace = orderBspace,
        pointsInSpace = pointsInSpace,
        pointsInTime = pointsInTime,
        supplementaryOrder = supplementaryOrder,
        fieldItpl = fieldItpl,
        materItpl = materItpl,
    )
    for orderBspace in orderBspace_list
    for pointsInSpace in pointsInSpace_list
    for supplementaryOrder in supplementaryOrder_list
    for fieldItpl in make_itpl_cases(pointsInSpace)
    for materItpl in make_itpl_cases(pointsInSpace)
]


# model construction

In [ ]:
using Symbolics,CairoMakie
CairoMakie.activate!()

logsOfHinverse = [1.0*i for i in 0:4]

cases=[]
#prefix="B"*string(tmpOrderBspace)*"_"*"w"*string(tmpWorderBspace)*"_"*string(tmpSupplementaryOrder)*"_"
prefix=""
L = 10.0*π

cases = push!(cases,(name=prefix*"homogeneous",u=cos(x),β=1.0))
cases = push!(cases,(name=prefix*"sameλ",u=cos(x),β=sin(x)+2))
cases = push!(cases,(name=prefix*"twiceλ",u=cos(x),β=sin(x/2) + 2))
cases = push!(cases,(name=prefix*"sameλ_shifted_π_3",u=cos(x),β=sin(x+π/3) + 2))
cases = push!(cases,(name=prefix*"λ_2",u=cos(x),β=cos(x).^2 + 1))
cases = push!(cases,(name=prefix*"parabols",u=cos(x),β=x^2+ 1))


@variables x
∂ = Differential(x)


misfit = Array{Float64,3}(undef,length(logsOfHinverse),length(cases),length(all_Configcases))


fig = Figure()
ax = Axis(fig[1, 1], xlabel="x", ylabel="model")
iH=5
for iCase ∈ eachindex(cases)
    name,_,β = cases[iCase]
    ΔxTry = exp(-logsOfHinverse[iH])
    Nx = Int(L÷ΔxTry) +1
    Δx = L/(Nx-1)
    X = [Δx * (i-1) for i ∈ range(1,Nx)]
    model=[Symbolics.value(substitute(β,Dict(x=>X[i]))) for i ∈ range(1,Nx)]
    lines!(ax, X, model)
end
display(fig)
Nx=nothing
Δx=nothing
iH=nothing
modelFamily=Array{Any,3}(undef,length(logsOfHinverse),length(cases),length(all_Configcases))
forceFamily=Array{Any,3}(undef,length(logsOfHinverse),length(cases),length(all_Configcases))
for iConfigUsed ∈ eachindex(all_Configcases), iCase ∈ eachindex(cases), iH ∈ eachindex(logsOfHinverse)
    name,T,β = cases[iCase]
    iExperiment = (iH=iH,iCase=iCase,iPointsUsed=iConfigUsed)

    ΔxTry = exp(-logsOfHinverse[iH])
    Nx = Int(L÷ΔxTry) +1
    Δx = L/(Nx-1)
    X = [Δx * (i-1) for i ∈ range(1,Nx)]
    modelName = name*string(Nx)
    models=[]
    model=[Symbolics.value(substitute(β,Dict(x=>X[i]))) for i ∈ range(1,Nx)]
    models=push!(models, model)
    modelPoints = (Nx)
    
    
    q = mySimplify(β*∂(T))
    qₓ = mySimplify(∂(q))
    symbols=(Nx=Nx,Δx=Δx,X=X,q=q,qₓ=qₓ,T=T,β=β)
    
    tmpModel = (models=models, modelName=modelName, modelPoints=modelPoints,Δ=(Δx),symbols=symbols)
    modelFamily[iH,iCase,iConfigUsed]=tmpModel

    force = [Symbolics.value(substitute(qₓ,Dict(x=>X[i]))) for i ∈ range(1,Nx)]
    sourceFull=reshape(force,Nx,1,1)
    forceFamily[iH,iCase,iConfigUsed]=sourceFull
   
end


    

# input parameters

In [ ]:

famousEquationType="1DpoissonHetero" 

In [ ]:
for iConfigUsed ∈ eachindex(all_Configcases), iCase ∈ eachindex(cases), iH ∈ eachindex(logsOfHinverse)
    name,T,β = cases[iCase]
    iExperiment = (iH=iH,iCase=iCase,iPointsUsed=iConfigUsed)
    if iH*iCase*iConfigUsed == 1
    @unpack Nx,Δx,X,q,qₓ,T,β = modelFamily[iH,iCase,iConfigUsed].symbols
    @unpack orderBtime, orderBspace, pointsInSpace, pointsInTime, supplementaryOrder, fieldItpl, materItpl = all_Configcases[iConfigUsed]
    concreteParametersForOPTConstruction = @strdict famousEquationType Δ=modelFamily[iH,iCase,iConfigUsed].Δ orderBtime orderBspace pointsInSpace pointsInTime supplementaryOrder fieldItpl materItpl
    optRec=myProduceOrLoad(makeOPTsemiSymbolic,concreteParametersForOPTConstruction,"semiSymbolic")
    params=@strdict optRec=optRec modelFam=modelFamily[iExperiment.iH,iExperiment.iCase,iExperiment.iPointsUsed] absorbingBoundaries=nothing maskedRegionInSpace=nothing
    numOpt=numericalOperatorConstruction(params)

    numOps = numOpt["numOperators"]
    prepared = prepareNumericalOperators(numOps)
    sourceFull=forceFamily[iExperiment.iH,iExperiment.iCase,iExperiment.iPointsUsed]
    syntheticData = timeMarchingSchemePrepared(
        prepared,
        1, # Nt
        modelFamily[iExperiment.iH,iExperiment.iCase,iExperiment.iPointsUsed].Δ,
        modelFamily[iExperiment.iH,iExperiment.iCase,iExperiment.iPointsUsed].modelName;
        videoMode=false,
        sourceType="Explicit",
        sourceFull=sourceFull,
        iExperiment=iExperiment,
        boundaryConditionForced=true,
    )
    syntheticData=reduce(vcat,syntheticData)
    
    analyticalData = [Symbolics.value(substitute(T,Dict(x=>X[i]))) for i ∈ range(1,Nx)]

    fig =Figure()
    ax=Axis(fig[1,1]; title="Comparison for h=$Δx, model=$(cases[iCase].name), points=1+$pointsInSpace")
    lines!(ax,X,analyticalData,color=:blue)
    scatter!(ax,X,syntheticData,color=:red,marker=:circle)        
    #axislegend(ax)
    display(fig)
    
    #save("plot_$iH_$iCase_$iPointsUsed.png",fig)
    #sleep(0.5)
    misfit[iH,iCase,iConfigUsed] = norm(syntheticData-analyticalData)


    end

end